# Week 5 — Cohort Analysis: Exploring Consumer Behavior Over Time

**Course:** Data Science for Business (YSU) · Customer Behavior Analysis  
**Prereq:** cleaned transactions from Weeks 1.2–1.4 (`data/data_cleared.csv`)  
**Next:** Week 6 — RFM segmentation

---

### Learning objectives

By the end of this lecture you should be able to:

1. Explain what a **cohort** is, and why totals hide the story.
2. Distinguish **acquisition** cohorts from **behavioral** cohorts.
3. Build a monthly retention table from raw transactions.
4. Read a cohort heatmap in two directions (customer lifetime vs calendar time).
5. Name the traps that make a heatmap lie: wrong grain, left-censoring, right-censoring, activity vs subscription.

### Agenda (~90 minutes)

| Block | What we do |
|---|---|
| 1. Why cohorts | A vanity-metric story |
| 2. Tiny example | Compute a 3-customer table by hand |
| 3. Real data | Customer–month grain, first purchase, period index |
| 4. Retention | Counts, %, heatmap, average curve |
| 5. Money | Spend of *active* customers, spend index |
| 6. Behavior | High vs low first-order value |
| 7. Practice | Exercises + bridge to RFM |

## 1. Why cohort analysis exists

A **cohort** is a group of people who share a defining event in the same time window — usually *the month they first bought* (or signed up, installed, deposited).

**Cohort analysis** tracks that *same group* forward in time. We stop asking “how many users do we have today?” and start asking “of the people who arrived in March, how many still buy in June?”

Pick one **key metric** before you start. Today that metric is **repeat-purchase retention**. Later we will also look at spend. The same skeleton works for churn, app opens, or GGR.

### The vanity-metric trap

Imagine a dashboard that says: *active customers are flat — the business is stable.*

A cohort table can show the opposite: every day a wave of new users arrives, tries the product for an hour, and never returns. Growth is filling a leaking bucket. The headline number never moved, so nobody noticed.

> Cohort analysis separates **growth** (how many arrived) from **engagement** (how many stayed).

That is why it sits in the **descriptive** layer of this course — but it is the descriptive tool that most often changes a product or marketing decision.

### Two types of cohorts

| Type | How we group people | Example question |
|---|---|---|
| **Acquisition (time)** | When they first appeared | Do 2021-03 customers retain better than 2020-12? |
| **Behavioral** | What they *did* | Do high first-order spenders come back more often? |

We will build both. Most “cohort heatmaps” you see in industry are acquisition cohorts. Behavioral cohorts are where the action usually is.

### Two ways to read the table

![How to read a cohort table](docs/cohort.png)

*Image: [CleverTap — Cohort Analysis](https://clevertap.com/blog/cohort-analysis/)*

1. **Along a row (customer lifetime).** One acquisition month, moving right: “What happens to the March cohort as they age?”
2. **Down a column (product / calendar time).** One age (e.g. month 2), different arrival months: “Is the product getting better at retaining people after 2 months?”

A third, quieter read: **diagonals** are the same calendar month. A Christmas spike will light up a diagonal, not a row. We will come back to that.

### Vocabulary we will use

| Term | Meaning in this lecture |
|---|---|
| **Acquisition month** | First *observed* purchase month of a customer |
| **Period 0** | That same month. Retention is 100% by construction |
| **Period $k$** | $k$ months later. “Active” = made ≥ 1 purchase that month |
| **Cohort size** | Unique customers in period 0 |
| **Retention** | Active customers in period $k$ ÷ cohort size |
| **Activity retention** | Retail version: they can skip a month and still count later |
| **Subscription retention** | SaaS version: once they cancel, they are gone (not our data) |

This dataset is **retail**, not a subscription. A customer who buys in January, skips February, and buys in March is *retained in period 2*. The heatmap can go **up**. That is not a bug.

## 2. Compute one table by hand first

Before pandas, lock the definition with three customers and four months.

| Customer | Purchases in |
|---|---|
| A | Jan, Feb, Apr |
| B | Jan only |
| C | Feb, Mar, Apr |

**Pause.** On paper:

- Who belongs to the January cohort? The February cohort?
- What is January’s retention in period 1? Period 2? Period 3?
- Is period 2 empty, or zero? Those are different.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

toy = pd.DataFrame(
    {
        "CustomerID": ["A", "A", "A", "B", "C", "C", "C"],
        "PurchaseMonth": pd.PeriodIndex(
            ["2021-01", "2021-02", "2021-04", "2021-01", "2021-02", "2021-03", "2021-04"],
            freq="M",
        ),
        "revenue": [40, 25, 30, 15, 80, 20, 25],
    }
)
toy["FirstPurchaseMonth"] = toy.groupby("CustomerID")["PurchaseMonth"].transform("min")
toy["PeriodNumber"] = (toy["PurchaseMonth"] - toy["FirstPurchaseMonth"]).apply(lambda p: p.n)
toy.sort_values(["FirstPurchaseMonth", "CustomerID", "PeriodNumber"])

In [ ]:
toy_counts = toy.pivot_table(
    index="FirstPurchaseMonth",
    columns="PeriodNumber",
    values="CustomerID",
    aggfunc="nunique",
)
toy_retention = toy_counts.divide(toy_counts.iloc[:, 0], axis=0)

print("Active customers")
display(toy_counts.fillna(""))
print("Retention")
display(toy_retention.style.format("{:.0%}", na_rep=""))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
sns.heatmap(toy_counts, annot=True, fmt=".0f", cmap="RdYlGn", ax=axes[0], linewidths=0.5)
axes[0].set_title("Toy: active customers")
sns.heatmap(toy_retention, annot=True, fmt=".0%", cmap="RdYlGn", ax=axes[1], linewidths=0.5, vmin=0, vmax=1)
axes[1].set_title("Toy: retention")
for ax in axes:
    ax.set_xlabel("Period (months since first purchase)")
    ax.set_ylabel("Acquisition cohort")
plt.tight_layout()

**What the toy table should have taught you**

- January cohort = {A, B}, size 2. February cohort = {C}, size 1.
- January period 1 = 50% (only A). Period 2 = **empty, not 0%** — nobody from January was even *observable as a purchase* in March, and in fact nobody bought. In a real pivot, a missing month becomes `NaN` and we mask it. If A had been active in March it would be 50% again.
- January period 3 = 50%: A **came back** after a quiet month. Retail retention is not a survival curve that can only fall.
- Period 0 is always 100%. If it is not, the cohort key is wrong.
- `PeriodNumber` is *relative age*, not the calendar month. C’s April purchase is period 2, not period 3.

## 3. Real data: get the grain right

We use the cleaned Online Retail file from earlier weeks. Each row is still a **line item** (one product on one invoice), not a customer and not an order.

Cohort analysis needs:

1. a stable **customer id**
2. a **timestamp**
3. (for money) a **spend** column

If `data/data_cleared.csv` is missing, the loader builds a synthetic retail file with the same columns so you can still run the lecture.

In [ ]:
def make_synthetic_retail(n_customers=1800, seed=5) -> pd.DataFrame:
    """Lecture fallback: enough structure to teach heatmaps, not a substitute for the course file."""
    rng = np.random.default_rng(seed)
    months = pd.period_range("2010-12", "2011-12", freq="M")
    first = rng.choice(months[:-1], size=n_customers, p=_acq_weights(len(months) - 1))
    rows = []
    invoice = 10000
    for i, start in enumerate(first):
        cid = 10000 + i
        quality = rng.uniform(0.15, 0.55)
        first_spend = float(rng.lognormal(3.4, 0.7))
        for m in months[months >= start]:
            age = (m - start).n
            if age == 0:
                p_buy = 1.0
            else:
                p_buy = quality * (0.72 ** max(age - 1, 0))
                p_buy *= 1.25 if m.month == 12 else 1.0
                p_buy = min(p_buy, 0.95)
            if rng.random() > p_buy:
                continue
            n_lines = int(rng.integers(1, 5))
            spend = first_spend if age == 0 else first_spend * rng.uniform(0.4, 1.3)
            for _ in range(n_lines):
                invoice += 1
                qty = int(rng.integers(1, 8))
                rows.append(
                    {
                        "InvoiceNo": invoice,
                        "InvoiceDate": m.to_timestamp() + pd.Timedelta(days=int(rng.integers(0, 27))),
                        "CustomerID": cid,
                        "Quantity": qty,
                        "TotalPrice": spend / n_lines,
                    }
                )
    return pd.DataFrame(rows)


def _acq_weights(n):
    w = np.linspace(1.4, 0.7, n)
    w[0] *= 1.6  # December 2010 is a large first month in the real file too
    return w / w.sum()


def load_transactions() -> pd.DataFrame:
    for path in [Path("data/data_cleared.csv"), Path("../data/data_cleared.csv")]:
        if path.exists():
            df = pd.read_csv(path)
            print(f"Loaded {len(df):,} line items from {path.resolve()}")
            return df
    print("data/data_cleared.csv not found — using synthetic retail data.")
    return make_synthetic_retail()


data = load_transactions()
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])
data["CustomerID"] = pd.to_numeric(data["CustomerID"], errors="coerce")
data = data.dropna(subset=["CustomerID", "InvoiceDate"]).copy()
data["CustomerID"] = data["CustomerID"].astype("int64")
data["PurchaseMonth"] = data["InvoiceDate"].dt.to_period("M")

print(data[["InvoiceNo", "InvoiceDate", "CustomerID", "Quantity", "TotalPrice", "PurchaseMonth"]].head())
print(
    f"\n{data['CustomerID'].nunique():,} customers | "
    f"{data['InvoiceNo'].nunique():,} invoices | "
    f"{data['PurchaseMonth'].min()} to {data['PurchaseMonth'].max()}"
)

### Collapse line items → one row per customer per month

If we count `InvoiceNo` on line items, we are counting *products*, not orders.  
If we take `mean(TotalPrice)` on line items, a customer who bought one cheap ribbon and one expensive set looks “average.”

The unit we want:

> **customer × month:** how many **orders**, how many **items**, how much **revenue**.

In [ ]:
customer_month = (
    data.groupby(["CustomerID", "PurchaseMonth"], as_index=False)
    .agg(
        n_orders=("InvoiceNo", "nunique"),
        n_items=("Quantity", "sum"),
        revenue=("TotalPrice", "sum"),
    )
    .sort_values(["PurchaseMonth", "CustomerID"])
)

customer_month["FirstPurchaseMonth"] = customer_month.groupby("CustomerID")["PurchaseMonth"].transform("min")
customer_month["PeriodNumber"] = (
    customer_month["PurchaseMonth"] - customer_month["FirstPurchaseMonth"]
).apply(lambda period: period.n)

print(customer_month.head(8))
print(f"\nCustomer–month rows: {len(customer_month):,}")
print("One customer, many months — look at the first id with more than one row:")
repeat_id = customer_month.loc[customer_month.duplicated("CustomerID", keep=False), "CustomerID"].iloc[0]
customer_month.loc[customer_month["CustomerID"] == repeat_id]

`PeriodNumber` comes from pandas `Period` arithmetic: `2011-03 − 2010-12` is a 3-month offset, and `.n` pulls out the integer `3`. Period 0 is the acquisition month.

**Left-censoring (say this out loud):** the first purchase *in this file* is not necessarily the customer’s first purchase *in life*. The file starts in December 2010. Anyone who bought in 2009 and again in January 2011 is labelled as a January 2011 “new” customer. We cannot fix that without older history. We just refuse to over-interpret the first one or two cohorts.

In [ ]:
active = customer_month.groupby("PurchaseMonth")["CustomerID"].nunique()
new_cust = (
    customer_month.loc[customer_month["PeriodNumber"].eq(0)]
    .groupby("PurchaseMonth")["CustomerID"]
    .nunique()
)

mix = pd.DataFrame({"active_customers": active, "new_customers": new_cust}).fillna(0)
mix["returning_customers"] = mix["active_customers"] - mix["new_customers"]

ax = mix[["new_customers", "returning_customers"]].plot(
    kind="bar", stacked=True, figsize=(10, 4), color=["#5b8def", "#3aa76d"]
)
ax.set_title("Active customers each month = new + returning")
ax.set_xlabel("Calendar month")
ax.set_ylabel("Customers")
ax.tick_params(axis="x", rotation=70)
plt.tight_layout()

mix

That stacked bar is the whole point of the lecture in one chart: the black line on a KPI dashboard is the *sum of the two colors*. If new (blue) is replacing returning (green), the business is not “stable.”

## 4. Acquisition cohorts and retention

Count **unique customers** in each (acquisition month, age) cell. Then divide every cell in a row by that row’s period-0 value.

In [ ]:
def cohort_pivot(frame, values, aggfunc):
    return frame.pivot_table(
        index="FirstPurchaseMonth",
        columns="PeriodNumber",
        values=values,
        aggfunc=aggfunc,
    )


n_customers = cohort_pivot(customer_month, "CustomerID", "nunique")
cohort_size = n_customers.iloc[:, 0]
retention = n_customers.divide(cohort_size, axis=0)

print("Cohort sizes (period 0)")
display(cohort_size.to_frame("cohort_size").T)
print("Active customers — first 8 periods")
display(n_customers.iloc[:, :8])
print("Retention — first 8 periods")
retention.iloc[:, :8].style.format("{:.1%}", na_rep="")

In [ ]:
def plot_cohort_heatmap(matrix, title, fmt, vmin=None, vmax=None, cbar_label=""):
    fig, ax = plt.subplots(figsize=(12, 6.5))
    sns.heatmap(
        matrix,
        mask=matrix.isna(),
        annot=True,
        fmt=fmt,
        cmap="RdYlGn",
        vmin=vmin,
        vmax=vmax,
        linewidths=0.35,
        ax=ax,
        cbar_kws={"label": cbar_label} if cbar_label else None,
    )
    ax.set_title(title)
    ax.set_xlabel("Months since first purchase  (0 = acquisition month)")
    ax.set_ylabel("Acquisition cohort")
    plt.tight_layout()
    return ax


plot_cohort_heatmap(
    n_customers,
    "Monthly acquisition cohorts — number of active customers",
    fmt=".0f",
    cbar_label="Customers",
);

In [ ]:
plot_cohort_heatmap(
    retention,
    "Monthly acquisition cohorts — repeat-purchase retention",
    fmt=".0%",
    vmin=0,
    vmax=0.5,
    cbar_label="Share of the original cohort that purchased again",
);

### How to read this heatmap in class

Walk these four questions on the live figure (do not skip them):

1. **Period 0.** Is every row 100%? If yes, the denominator is correct.
2. **The cliff into period 1.** Most retail datasets lose more than half the cohort immediately. That is an onboarding / first-experience problem, not a “long-term loyalty” problem.
3. **A single row, left to right.** Does retention flatten (a loyal core) or keep sliding (slow leak)? Does it bounce up (reactivation / seasonality)?
4. **A single column, top to bottom.** Later cohorts better than earlier ones at the same age → the product or acquisition mix improved. Worse → we are buying cheaper, less loyal traffic.

### Two ways the last columns lie

- **Right-censoring.** A customer who first bought in November 2011 can only be observed for 0–1 periods if the file ends in December. Empty cells in the bottom-right are *not* churn. Do not average them as zeros.
- **Seasonality on the diagonal.** A November campaign or Christmas will raise *every* living cohort in the same calendar month. That looks like “retention improved with age” if you only read rows.

In [ ]:
# Mean retention by age. Later periods rest on fewer, older cohorts — so the tail is fragile.
mean_retention = retention.mean(axis=0)
n_cohorts_seen = retention.notna().sum(axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(mean_retention.index, mean_retention.values, marker="o")
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(lambda x, _pos: f"{x:.0%}")
ax.set_title("Average retention curve (mean across cohorts that have that period)")
ax.set_xlabel("Period")
ax.set_ylabel("Retention")
ax.set_xticks(mean_retention.index)

ax2 = ax.twinx()
ax2.bar(n_cohorts_seen.index, n_cohorts_seen.values, alpha=0.18, color="#444")
ax2.set_ylabel("Cohorts contributing to the average")
plt.tight_layout()

pd.DataFrame({"mean_retention": mean_retention, "n_cohorts": n_cohorts_seen}).T.style.format(
    "{:.1%}", subset=pd.IndexSlice["mean_retention", :]
)

## 5. Money: spend of people who actually bought

Customer retention can fall while **revenue per remaining customer** rises — the people who stay are the ones who spend. Or the opposite: lots of people come back and buy one cheap item.

We look at two related numbers:

| Metric | Formula | Reads as |
|---|---|---|
| **Average spend among actives** | mean revenue of customer–months in that cell | “How big is a typical returning basket?” |
| **Spend index** | that mean ÷ the cohort’s period-0 mean | “Are returning trips larger or smaller than the first?” |

A spend index is **not** a retention rate. 140% means returning buyers spent more than they did in month 0, not that 140% of people returned.

In [ ]:
avg_spend = cohort_pivot(customer_month, "revenue", "mean")
spend_index = avg_spend.divide(avg_spend.iloc[:, 0], axis=0)

plot_cohort_heatmap(
    avg_spend,
    "Average spend among active customers (£)",
    fmt=".0f",
    cbar_label="Mean revenue in that customer–month",
);

In [ ]:
plot_cohort_heatmap(
    spend_index,
    "Spend index vs first month  (1.0 = same as acquisition month)",
    fmt=".0%",
    vmin=0.5,
    vmax=1.5,
    cbar_label="Mean spend / cohort's period-0 mean spend",
);

**Optional extra (if there is time):** *revenue retention* =
total cohort revenue in period $k$ ÷ total cohort revenue in period 0.

It mixes “how many came back” with “how much they spent.” Useful for finance; easier to misread than the two charts above. If you show it, always pair it with headcount retention.

In [ ]:
total_revenue = cohort_pivot(customer_month, "revenue", "sum")
revenue_retention = total_revenue.divide(total_revenue.iloc[:, 0], axis=0)

compare = pd.DataFrame(
    {
        "customer_retention_p1": retention[1],
        "revenue_retention_p1": revenue_retention[1],
    }
).dropna()
compare["gap"] = compare["revenue_retention_p1"] - compare["customer_retention_p1"]
print("Period 1: if gap > 0, returning customers are more valuable than the average newcomer.")
compare.style.format("{:.1%}")

## 6. Behavioral cohorts — the type the intro promised

Acquisition month is a convenient label, not a cause. Split the *same* customers by something they **did**: first-month spend above vs below the median.

Question: *do bigger first orders predict higher repeat rates, or just one expensive visit?*

In [ ]:
first_month = customer_month.loc[customer_month["PeriodNumber"].eq(0), ["CustomerID", "revenue"]].rename(
    columns={"revenue": "first_month_revenue"}
)
cut = first_month["first_month_revenue"].median()
first_month["first_spend_group"] = np.where(
    first_month["first_month_revenue"] >= cut, "high first spend", "low first spend"
)

labeled = customer_month.merge(first_month[["CustomerID", "first_spend_group"]], on="CustomerID", how="left")

behavior = (
    labeled.groupby(["first_spend_group", "PeriodNumber"], as_index=False)["CustomerID"]
    .nunique()
    .pivot(index="first_spend_group", columns="PeriodNumber", values="CustomerID")
)
behavior_retention = behavior.divide(behavior.iloc[:, 0], axis=0)

print(f"Median first-month spend: {cut:,.2f}")
display(behavior_retention.iloc[:, :8].style.format("{:.1%}", na_rep=""))

fig, ax = plt.subplots(figsize=(9, 4))
for group, row in behavior_retention.iterrows():
    ax.plot(row.index, row.values, marker="o", label=group)
ax.yaxis.set_major_formatter(lambda x, _pos: f"{x:.0%}")
ax.set_ylim(0, 1.05)
ax.set_title("Behavioral cohort: retention by first-month spend")
ax.set_xlabel("Period")
ax.set_ylabel("Retention")
ax.legend()
plt.tight_layout()

Treat the split as a **hypothesis generator**, not a causal claim. High first spend might mean a wholesale buyer, a gift-season basket, or a customer who already trusted the brand. Next week’s RFM scores will cut the same people by recency, frequency, and monetary value — a different lens on the same idea.

## 7. Mistakes that produce a confident, wrong heatmap

| Mistake | What happens | What to do instead |
|---|---|---|
| Count line items or raw `InvoiceNo` rows | Inflates the number of “customers” | `nunique` on `CustomerID` after a customer–month collapse |
| `mean(TotalPrice)` on line items | One cheap SKU drags the average down | Sum to customer–month first, then take the mean |
| Treat `NaN` as 0% | Last cohorts look like they all churned | Mask missing periods; do not fill |
| Read a Christmas diagonal as “loyalty grows with age” | Seasonality impersonates retention | Check which cells share a calendar month |
| Call the first file month “true acquisition” | Old customers are relabelled as new | Name it *first observed* month |
| Mix SaaS language with retail data | “Churned in February” is false if they return in April | Say *active / inactive that month* |

Period 0 ≠ 100% is the fastest unit test you have. If you see it, stop and fix the join.

## 8. Where this sits in the course

| Week | Question | Grain |
|---|---|---|
| **5 · Cohorts** | How do *groups* evolve after they arrive? | customer × time |
| **6 · RFM** | Who is valuable *right now*? | one row per customer |
| **7–8 · Clustering** | Are there natural segments beyond RFM rules? | customer features |
| **10 · Churn** | Who is *about to leave*? | predictive label |

Cohorts tell you whether the *machine* is getting better. RFM tells you whom to call this week. You need both.

## 9. Exercises (do these before you leave)

**E1. Read a number.**  
What is period-1 retention for the largest cohort? Write the percentage and one business hypothesis (product, season, or acquisition channel — you may not have channel in the file; that is fine, state the assumption).

**E2. Change the grain.**  
Rebuild the retention heatmap with **quarters** instead of months (`dt.to_period("Q")`). Does the period-1 cliff look milder? Why would that happen even if customer behavior did not change?

**E3. Another behavioral cut.**  
Split customers by whether their first month had **one order** or **more than one**. Compare period-1 retention. Which group should a marketer try to create on purpose?

**E4 (stretch).**  
If `Country` is in the file, compare the UK retention curve with everyone else. Watch sample size: a pretty line on 40 customers is not a finding.

In [ ]:
# E1 — largest cohort, period-1 retention
largest = cohort_size.idxmax()
print(f"Largest cohort: {largest}  (n = {int(cohort_size.loc[largest])})")
print(f"Period-1 retention: {retention.loc[largest, 1]:.1%}")

# E2 — your turn: copy the pipeline with PurchaseQuarter = InvoiceDate.dt.to_period("Q")

# E3 — your turn: first_month_orders = customer_month.query("PeriodNumber == 0")["n_orders"]

### Solution sketches (after you try)

<details>
<summary>E2 — why quarterly retention looks “better”</summary>

A customer who buys once in January and once in March is **inactive** in monthly period 1 (February) but **active** in quarterly period 0 or 1, depending on the cut. Coarser bins hide gaps. Always name the bin when you quote a retention number.

</details>

<details>
<summary>E3 — starter code</summary>

```python
first_orders = customer_month.query("PeriodNumber == 0")[["CustomerID", "n_orders"]]
first_orders["multi_order"] = np.where(first_orders["n_orders"] > 1, "2+ orders in month 0", "1 order in month 0")
labeled2 = customer_month.merge(first_orders[["CustomerID", "multi_order"]], on="CustomerID")
tab = labeled2.pivot_table(index="multi_order", columns="PeriodNumber", values="CustomerID", aggfunc="nunique")
(tab.T / tab.iloc[:, 0]).T.iloc[:, :6]
```

</details>

## 10. Takeaways

1. A cohort is a group that shares an event. Acquisition month is the default event; behavior is often the better one.
2. Retention = actives at age $k$ ÷ size at age 0. Period 0 must be 100%.
3. In retail, people can skip months. The heatmap is *activity*, not a one-way survival curve.
4. Get the grain right: line item → customer–month → (cohort, period).
5. Empty cells in the bottom-right are incomplete history, not 0% retention.
6. Always pair headcount retention with some view of spend. The customers who stay are rarely the average customer.

### Further reading

- [CleverTap — Cohort Analysis](https://clevertap.com/blog/cohort-analysis/) (the figure we used)
- [Amplitude — Mastering Retention](https://amplitude.com/blog/how-to-calculate-retention) (product-style retention vs revenue retention)
- Next lecture: [Week 6 — RFM](https://github.com/armhijacker/customer_behaviour/blob/master/Week_6_Basics_of_Segmentation_RFM.ipynb)